# Paper 2 · Analisis tambahan (CPU)

Notebook ini **tidak butuh GPU** dan tidak melatih apa pun. Ia hanya membaca file hasil langkah 4, 5, dan 6 yang sudah ada di Drive.

1. **Selang kepercayaan uji deletion.** Sebelumnya peta dinyatakan "lulus" hanya dari perbandingan rata-rata kelompok. Di sini dihitung keunggulan peta atas kontrol terbaik per gambar, dengan selang kepercayaan bootstrap 95% (saran Kuturin dkk., 2026).
2. **Uji lintas generator per gaya.** Berapa persen gambar AI dari generator yang tidak dikenal tetap dikenali, dipecah per gaya lukisan, untuk dibandingkan dengan pola gaya di Thakur dkk. (2026).
3. **Elemen desain per gaya untuk gambar yang lolos**, dengan peta skala logit.

**Sebelum Run all:** menu **Runtime → Change runtime type → CPU → Save**. Selesai dalam beberapa menit.

## 1. Persiapan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import time, platform
import numpy as np, pandas as pd, scipy
from pathlib import Path
from scipy import stats

SEED = 42
N_BOOT = 2000
PROJECT = Path('/content/drive/MyDrive/Paper2')
NOTES, RESULTS = PROJECT / 'catatan', PROJECT / 'hasil'
for f in ('rq2_peta_per_gambar.csv', 'rq2_elemen_logit_per_gambar.csv'):
    assert (RESULTS / f).exists(), f'File {f} belum ada. Jalankan notebook langkah 4 dan 6 dulu.'
env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\nnumpy: {np.__version__}\n'
       f'pandas: {pd.__version__}\nscipy: {scipy.__version__}\nbootstrap: {N_BOOT} ulangan, seed {SEED}\n')
(NOTES / 'lingkungan_analisis_tambahan.txt').write_text(env)
print(env)

## 2. Selang kepercayaan uji deletion

Untuk setiap gambar dihitung **keunggulan** = luas kurva kontrol terbaik dikurangi luas kurva peta. Positif berarti peta lebih setia daripada kontrol. Kontrol terbaik adalah yang rata-ratanya paling kecil di kelompok itu (tengah atau acak), jadi pembandingnya selalu yang paling sulit dikalahkan. Peta dinyatakan unggul hanya kalau batas bawah selang kepercayaan 95% di atas nol.

Kolom peta: **occ-logit** (peta utama, langkah 6), **occ-prob** (langkah 4), **grad-cam**.

In [ ]:
new = pd.read_csv(RESULTS / 'rq2_elemen_logit_per_gambar.csv')
old = pd.read_csv(RESULTS / 'rq2_peta_per_gambar.csv')
UNITS = new.bagian.unique().tolist()
old = old[old.bagian.isin(UNITS)][['bagian', 'sample_id'] + [f'del_{r}_{m}' for r in ('blur', 'mean') for m in ('gc', 'occ', 'centre', 'random')]]
dd = new.merge(old, on=['bagian', 'sample_id'], how='left')
MAIN = ['5-fold: manusia benar', '5-fold: LD benar', '5-fold: SD benar', 'lintas: LD lolos', 'lintas: SD lolos', 'lintas: manusia benar']
MAPS = {'occ-logit': 'occlogit', 'occ-prob': 'occ', 'grad-cam': 'gc'}
rng = np.random.default_rng(SEED)

def boot_ci(x):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]; n = len(x)
    if n < 10:
        return np.nan, np.nan, np.nan, n
    idx = rng.integers(0, n, size=(N_BOOT, n))
    m = x[idx].mean(1)
    return x.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5), n

rows = []
for grp in MAIN:
    g = dd[dd.kelompok == grp]
    if len(g) == 0:
        continue
    for r in ('blur', 'mean'):
        ctrl = min(('centre', 'random'), key=lambda c: g[f'del_{r}_{c}'].mean())
        for label, col in MAPS.items():
            diff = g[f'del_{r}_{ctrl}'] - g[f'del_{r}_{col}']
            mean, lo, hi, n = boot_ci(diff)
            rows.append({'kelompok': grp, 'pengganti': r, 'peta': label, 'kontrol_terbaik': 'tengah' if ctrl == 'centre' else 'acak',
                         'n': n, 'keunggulan': mean, 'ci_bawah': lo, 'ci_atas': hi,
                         'unggul': bool(lo > 0) if not np.isnan(lo) else False,
                         'persen_gambar_peta_lebih_baik': float((diff > 0).mean()) if n else np.nan})
B = pd.DataFrame(rows); B.to_csv(RESULTS / 'rq2_deletion_selang_kepercayaan.csv', index=False)

def fmt(r):
    if pd.isna(r.keunggulan): return 'nan'
    return f'{r.keunggulan:+.3f} [{r.ci_bawah:+.3f}, {r.ci_atas:+.3f}]' + (' *' if r.unggul else '  ')
B['sel'] = B.apply(fmt, axis=1)
for r in ('blur', 'mean'):
    t = B[B.pengganti == r].pivot(index='kelompok', columns='peta', values='sel').reindex([g for g in MAIN if g in B.kelompok.values])[list(MAPS)]
    k = B[(B.pengganti == r) & (B.peta == 'occ-logit')].set_index('kelompok').kontrol_terbaik
    t['kontrol terbaik'] = k
    print(f"\nB-{r}. KEUNGGULAN PETA ATAS KONTROL TERBAIK, pengganti = {'blur' if r == 'blur' else 'warna rata-rata'}")
    print('   rata-rata [selang 95%]; * = batas bawah di atas nol; positif = peta lebih setia')
    print(t.to_string())

## 3. Uji lintas generator per gaya

Model **riwayat_disamakan** yang dilatih pada satu generator diuji pada generator lain. Tabel ini memecah hasilnya per gaya: berapa persen gambar AI tetap dikenali sebagai AI, dan berapa persen karya manusia tetap dikenali sebagai manusia.

In [ ]:
lin = new[new.lintas].copy()
lin['benar'] = lin.pred == lin.y
SRC = {'human': 'manusia', 'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
NAMA = {'model-LD': 'dilatih LD, diuji SD', 'model-SD': 'dilatih SD, diuji LD'}
rows = []
for mdl, g in lin.groupby('model'):
    ai, hu = g[g.y == 1], g[g.y == 0]
    rows.append({'model': NAMA.get(mdl, mdl), 'gaya': 'SEMUA', 'n_AI': len(ai), 'AI_dikenali_%': 100 * ai.benar.mean(),
                 'n_manusia': len(hu), 'manusia_benar_%': 100 * hu.benar.mean() if len(hu) else np.nan})
    for st, s in g.groupby('style'):
        a, h = s[s.y == 1], s[s.y == 0]
        rows.append({'model': NAMA.get(mdl, mdl), 'gaya': st, 'n_AI': len(a), 'AI_dikenali_%': 100 * a.benar.mean() if len(a) else np.nan,
                     'n_manusia': len(h), 'manusia_benar_%': 100 * h.benar.mean() if len(h) else np.nan})
G = pd.DataFrame(rows); G.to_csv(RESULTS / 'rq3_lintas_per_gaya.csv', index=False)
for mdl, t in G.groupby('model', sort=False):
    t = pd.concat([t[t.gaya == 'SEMUA'], t[t.gaya != 'SEMUA'].sort_values('AI_dikenali_%', ascending=False)])
    print(f'\nG. {mdl.upper()}  (diurutkan dari gaya yang paling mudah dikenali)')
    print(t.drop(columns='model').round(1).to_string(index=False))
print('\nCek: baris SEMUA harus sama dengan langkah 3 (dilatih LD, diuji SD: sekitar 4,9%; dilatih SD, diuji LD: sekitar 3,5%).')
print('Catatan: tiap gaya hanya sekitar 75 gambar AI, jadi selisih 1 sampai 2 gambar sudah mengubah persentase 1,3 sampai 2,7 poin.')

## 4. Elemen desain per gaya untuk gambar yang lolos (peta logit, area 20%)

Sama dengan langkah 5 dan 6 (rank-biserial r, kontrol acak-tukar, koreksi FDR 5% atas semua sel), tapi dipisah per gaya. Jumlah gambar per sel kecil, jadi tabel ini untuk melihat **apakah polanya konsisten di banyak gaya**, bukan untuk membaca satu sel.

In [ ]:
MEASURES = ['hue_entropy', 'saturation', 'colourfulness', 'brightness', 'contrast', 'gradient',
            'edge_density', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation']
LABEL = {'hue_entropy': 'Hue: entropi', 'saturation': 'Warna: saturasi', 'colourfulness': 'Warna: kekayaan',
         'brightness': 'Nada: kecerahan', 'contrast': 'Nada: kontras', 'gradient': 'Nada: gradasi',
         'edge_density': 'Garis: kepadatan tepi', 'glcm_contrast': 'Tekstur: kontras',
         'glcm_homogeneity': 'Tekstur: homogenitas', 'glcm_energy': 'Tekstur: energi', 'glcm_correlation': 'Tekstur: korelasi'}

def paired_effect(x):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]; x = x[x != 0]
    if len(x) < 10:
        return np.nan, np.nan, len(x)
    r = stats.rankdata(np.abs(x)); pos_, neg_ = r[x > 0].sum(), r[x < 0].sum()
    return (pos_ - neg_) / (pos_ + neg_), stats.wilcoxon(x).pvalue, len(x)
def bh(p):
    p = np.asarray(p, dtype=float); out = np.full(len(p), np.nan); ok = ~np.isnan(p)
    if ok.sum() == 0: return out
    pv = p[ok]; order = np.argsort(pv); ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]; res = np.empty(len(pv)); res[order] = np.minimum(ranked, 1)
    out[ok] = res; return out

GROUPS = {'LD lolos (model dilatih SD)': (new.kelompok == 'lintas: LD lolos'),
          'SD lolos (model dilatih LD)': (new.kelompok == 'lintas: SD lolos'),
          'manusia benar (model dilatih SD)': (new.kelompok == 'lintas: manusia benar') & (new.model == 'model-SD')}
rows = []
for gname, mask in GROUPS.items():
    d = new[mask & ~new.peta_kosong]
    for st, s in d.groupby('style'):
        for m in MEASURES:
            r, p, n = paired_effect((s[f'in_{m}'] - s[f'out_{m}']) - (s[f'shin_{m}'] - s[f'shout_{m}']))
            rows.append({'kelompok': gname, 'gaya': st, 'ukuran': m, 'n': n, 'r': r, 'p': p})
S = pd.DataFrame(rows); S['p_fdr'] = bh(S.p.values); S['bermakna'] = S.p_fdr < 0.05
S.to_csv(RESULTS / 'rq3_elemen_lolos_per_gaya.csv', index=False)

SHOW = ['colourfulness', 'gradient', 'edge_density', 'glcm_contrast', 'glcm_homogeneity']
for gname in GROUPS:
    t = S[(S.kelompok == gname) & S.ukuran.isin(SHOW)].copy()
    if t.empty:
        continue
    t['sel'] = [('nan' if pd.isna(a) else f'{a:+.2f}') + ('*' if b else ' ') for a, b in zip(t.r, t.bermakna)]
    pv = t.pivot(index='gaya', columns='ukuran', values='sel')[SHOW]; pv.columns = [LABEL[c] for c in pv.columns]
    pv['n'] = t.groupby('gaya').n.max()
    print(f'\nE. {gname.upper()}: r per gaya (* = bermakna setelah FDR atas semua sel)')
    print(pv.to_string())
    k = S[S.kelompok == gname].groupby('ukuran').apply(lambda x: pd.Series({'gaya_positif': int((x.r > 0).sum()), 'gaya_negatif': int((x.r < 0).sum())}))
    k.index = [LABEL[i] for i in k.index]
    print('   Konsistensi arah di 10 gaya (jumlah gaya dengan r positif | negatif):')
    print(k.reindex([LABEL[m] for m in MEASURES]).to_string())
print('\nTersimpan di', RESULTS)

## Selesai

Kalau tabel B-blur, B-mean, G, dan E muncul tanpa error, semua beres. Tekan **Cmd+S**, lalu kabari Claude.